<a href="https://colab.research.google.com/github/tejalkv0/RAG-College-Bot/blob/main/RAG_Collegebot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install  langchain_community

In [ ]:
!pip install langchain_huggingface

In [ ]:
!pip install faiss-gpu

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs = [
    "CIT established in 1999, is sponsored by the Kodava Education Society, with the primary aim of providing quality technical education to the aspiring students from Kodagu district and other parts of the country. A Governing Council headed by Sri. C.P. Belliappa, an engineer and coffee planter, manages CIT. The Governing Council comprises of professionals with extensive experience in the management of educational institutions. It leads to the implementation of comprehensive strategic plans that aims at leveraging the institute’s existing strengths and ensures its rapid progress into a Deemed University. The Governing Council meets periodically to frame policies and programmes for the growth of CIT. Dr. M.BASAVARAJ, Principal, oversees the academic administration of the college. "
]
# Crucial Step: Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = ["https://www.citcoorg.edu.in/","https://www.citcoorg.edu.in/computer-science-engineering/","https://www.citcoorg.edu.in/placements/","https://www.citcoorg.edu.in/library-2/"]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "Hacknova1.0-Brochure.pdf" # Instruct students to upload a PDF with this name
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("Genai")
)

print("\nɕ Unified Universal Bot Online. Ask about the crime scene, college, or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context.
    If the context does not contain the answer, say "I do not have enough information."
    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

Step 1: Ingesting data from multiple sources...
PDF loaded successfully.
Total raw pages/documents loaded: 14
Data chopped into 158 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


ɕ Unified Universal Bot Online. Ask about the crime scene, college, or PDF.
------------------------------------------------------------
AI: The hackathon focuses on the following domains:

- **Artificial Intelligence (AI)**
- **Internet of Things (IoT)**
- **Robotics**
- **Automation** (often grouped together as “Robotics & Automation”)


In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.2 MB/s eta 0:00:00
